In [1]:
%matplotlib widget


# HMI PFSS solutions
Calculating a PFSS solution from a HMI synoptic map.

This example shows how to calcualte a PFSS solution from a HMI synoptic map.
There are a couple of important things that this example shows:

- HMI maps have non-standard metadata, so this needs to be fixed
- HMI synoptic maps are very big (1440 x 3600), so need to be downsampled
  in order to calculate the PFSS solution in a reasonable time.


In [2]:
import os

import astropy.units as u
import matplotlib.pyplot as plt
import sunpy.map
from sunpy.net import Fido
from sunpy.net import attrs as a

import pfsspy
import pfsspy.utils

Set up the search.

Note that for SunPy versions earlier than 2.0, a time attribute is needed to
do the search, even if (in this case) it isn't used, as the synoptic maps are
labelled by Carrington rotation number instead of time



In [3]:
time = a.Time('2010/01/01', '2010/01/01')
series = a.jsoc.Series('hmi.synoptic_mr_polfil_720s')
#series = a.jsoc.Series('hmi.synoptic_mr_720s')
crot = a.jsoc.PrimeKey('CAR_ROT', 2240)

Do the search.

If you use this code, please replace this email address
with your own one, registered here:
http://jsoc.stanford.edu/ajax/register_email.html



In [4]:
result = Fido.search(time, series, crot,
                     a.jsoc.Notify("loeschl@mps.mpg.de"))
files = Fido.fetch(result)

Export request pending. [id=JSOC_20220922_3269_X_IN, status=2]
Waiting for 0 seconds...
1 URLs found for download. Full request totalling 4MB


Files Downloaded:   0%|          | 0/1 [00:00<?, ?file/s]

hmi.synoptic_mr_polfil_720s.2240.Mr_polfil.fits:   0%|          | 0.00/4.29M [00:00<?, ?B/s]

Read in a file. This will read in the first file downloaded to a sunpy Map
object



In [5]:
files

['/home/loeschl/sunpy/data/hmi.synoptic_mr_polfil_720s.2240.Mr_polfil.fits']

In [6]:
hmi_map = sunpy.map.Map(files[0])
print('Data shape: ', hmi_map.data.shape)

Data shape:  (1440, 3600)


Since this map is far to big to calculate a PFSS solution quickly, lets
resample it down to a smaller size.



In [7]:
hmi_map = hmi_map.resample([360, 180] * u.pix)
print('New shape: ', hmi_map.data.shape)

New shape:  (180, 360)


Now calculate the PFSS solution



In [8]:
nrho = 35
rss = 2.5
pfss_in = pfsspy.Input(hmi_map, nrho, rss)
pfss_out = pfsspy.pfss(pfss_in)

Using the Output object we can plot the source surface field, and the
polarity inversion line.



In [9]:
ss_br = pfss_out.source_surface_br
# Create the figure and axes
fig = plt.figure()
ax = plt.subplot(projection=ss_br)

# Plot the source surface map
ss_br.plot()
# Plot the polarity inversion line
ax.plot_coord(pfss_out.source_surface_pils[0])
# Plot formatting
plt.colorbar()
ax.set_title('Source surface magnetic field')

plt.show()

/scratch/slam/loeschl/conda/py38/lib/python3.8/site-packages/pfsspy/output.py:95: UserWarning: Could not parse unit string "Mx/cm^2" as a valid FITS unit.
See https://fits.gsfc.nasa.gov/fits_standard.html for the FITS unit standards.
  warnings.warn(f'Could not parse unit string "{unit_str}" as a valid FITS unit.\n'
  'degree' -> 'deg'. [astropy.wcs.wcs]
/scratch/slam/loeschl/conda/py38/lib/python3.8/site-packages/sunpy/util/decorators.py:378: SunpyMetadataWarning: Missing metadata for observer: assuming Earth-based observer.

  new_val = prop(instance)


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

/scratch/slam/loeschl/conda/py38/lib/python3.8/site-packages/pfsspy/output.py:95: UserWarning: Could not parse unit string "Mx/cm^2" as a valid FITS unit.
See https://fits.gsfc.nasa.gov/fits_standard.html for the FITS unit standards.
  warnings.warn(f'Could not parse unit string "{unit_str}" as a valid FITS unit.\n'


# Plot Open Closed Map

In [10]:
import astropy.constants as const

import matplotlib.colors as mcolor
import matplotlib.pyplot as plt
import numpy as np

from astropy.coordinates import SkyCoord

from pfsspy import tracing
from pfsspy.sample_data import get_gong_map

In [11]:
nrho = 40
rss = 2.5

Construct the input, and calculate the output solution



In [12]:
pfss_in = pfsspy.Input(hmi_map, nrho, rss)
pfss_out = pfsspy.pfss(pfss_in)

Finally, using the 3D magnetic field solution we can trace some field lines.
In this case a grid of 90 x 180 points equally gridded in theta and phi are
chosen and traced from the source surface outwards.

First, set up the tracing seeds



In [13]:
r = const.R_sun
# Number of steps in cos(latitude)
nsteps = 45
lon_1d = np.linspace(0, 2 * np.pi, nsteps * 2 + 1)
lat_1d = np.arcsin(np.linspace(-1, 1, nsteps + 1))
lon, lat = np.meshgrid(lon_1d, lat_1d, indexing='ij')
lon, lat = lon*u.rad, lat*u.rad
seeds = SkyCoord(lon.ravel(), lat.ravel(), r, frame=pfss_out.coordinate_frame)

  'degree' -> 'deg'. [astropy.wcs.wcs]
/scratch/slam/loeschl/conda/py38/lib/python3.8/site-packages/sunpy/util/decorators.py:378: SunpyMetadataWarning: Missing metadata for observer: assuming Earth-based observer.

  new_val = prop(instance)


Trace the field lines



In [14]:
print('Tracing field lines...')
tracer = tracing.FortranTracer(max_steps=2000)
field_lines = tracer.trace(seeds, pfss_out)
print('Finished tracing field lines')

Tracing field lines...


/scratch/slam/loeschl/conda/py38/lib/python3.8/site-packages/pfsspy/output.py:95: UserWarning: Could not parse unit string "Mx/cm^2" as a valid FITS unit.
See https://fits.gsfc.nasa.gov/fits_standard.html for the FITS unit standards.
  warnings.warn(f'Could not parse unit string "{unit_str}" as a valid FITS unit.\n'
/scratch/slam/loeschl/conda/py38/lib/python3.8/site-packages/pfsspy/tracing.py:180: UserWarning: At least one field line ran out of steps during tracing.
You should probably increase max_steps (currently set to 2000) and try again.
  warnings.warn(


Finished tracing field lines


Plot the result. The to plot is the input magnetogram, and the bottom plot
shows a contour map of the the footpoint polarities, which are +/- 1 for open
field regions and 0 for closed field regions.



In [15]:
fig = plt.figure()
m = pfss_in.map
ax = fig.add_subplot(2, 1, 1, projection=m)
m.plot()
ax.set_title('Input HMI magnetogram')

ax = fig.add_subplot(2, 1, 2)
cmap = mcolor.ListedColormap(['tab:red', 'black', 'tab:blue'])
norm = mcolor.BoundaryNorm([-1.5, -0.5, 0.5, 1.5], ncolors=3)
pols = field_lines.polarities.reshape(2 * nsteps + 1, nsteps + 1).T
ax.contourf(np.rad2deg(lon_1d), np.sin(lat_1d), pols, norm=norm, cmap=cmap)
ax.set_ylabel('sin(latitude)')

ax.set_title('Open (blue/red) and closed (black) field')
ax.set_aspect(0.5 * 360 / 2)

plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …